# Scanpy Extra: (For quick testing)

In [37]:
# This cell is labelled 'paramters' to work with papermill remotely
# Papermill will overwite the default local plate value below with whatever is passed to 
# the -p flag in the snakerule shell script
#os.system("conda activate eqtl_study") use this locally if using VScode
plate = 'plate1'
plate = globals().get("plate")
print(f"Processing plate: {plate}")

Processing plate: plate1


### Set Variables

In [ ]:
# Import custom utility packages, lists and functions
import sys
import os
if os.path.exists('/scratch/'):
    root_dir = '/scratch/SCWF00021/eQTL_study_2025/'
else:
    root_dir = '/Users/darren/Desktop/eQTL_study_2025/'
        
sys.path.append(root_dir + 'workflow/scripts/')

from scanpy_init_env import *
from scanpy_utils import *
from scanpy_gene_lists import *



### Pull out DE genes for all clusters

In [ ]:

# Load L1 clusters
logger, root_dir, sheets_dir, plate_path, scanpy_dir, report_dir = initialize_env(plate)

logger.info("Loading L1 data ...")
adata = sc.read(scanpy_dir + 'adata_clusters_v3.h5ad')

if 'cell_type' not in adata.obs.columns:
    logger.info("Creating 'cell_type' column from leiden_harmony_0.2 ...")

    if 'leiden_harmony_0.2' not in adata.obs.columns:
        raise KeyError("Column 'leiden_harmony_0.2' not found in adata.obs")

    adata.obs['cell_type'] = (
        adata.obs['leiden_harmony_0.2']
        .astype(str)
        .map(cluster_anns)
    )
else:
    logger.info("'cell_type' column already exists — using existing values")

print("Unmapped L1 cells:", adata.obs['cell_type'].isna().sum())
print("L1 cell_type counts:")
print(adata.obs['cell_type'].value_counts())

logger.info("Ranking L1 differentially expressed genes ...")
sc.tl.rank_genes_groups(
    adata,
    groupby='cell_type',
    method='t-test_overestim_var',
    key_added='L1_DE'
)

result = adata.uns['L1_DE']
groups = result['names'].dtype.names

print("L1 raw exists:", adata.raw is not None)
print("First logFC values:",
      result['logfoldchanges'][groups[0]][:5])

gene_tables = {}

for group in groups:
    gene_tables[f"{group}"] = pd.DataFrame({
        'gene': result['names'][group],
        'logfoldchange': result['logfoldchanges'][group],
        'pval': result['pvals'][group],
        'pval_adj': result['pvals_adj'][group],
        'score': result['scores'][group],
    })

# Free memory
del adata
gc.collect()


# Load L2 clusters
logger.info("Processing L2 subclusters ...")
subcluster_dir = scanpy_dir + 'subclustering/'

cell_types = ['Glu-UL', 'Glu-DL', 'NPC', 'GABA']

for cell_type in cell_types:

    logger.info(f"Loading L2 data for {cell_type} ...")
    adata_sub = sc.read(
        subcluster_dir + f'adata_{cell_type}_subclusters.h5ad'
    )

    if 'subcluster' not in adata_sub.obs.columns:
        raise KeyError(
            f"'subcluster' column missing in L2 object for {cell_type}"
        )

    logger.info(f"Ranking L2 DE genes for {cell_type} ...")
    sc.tl.rank_genes_groups(
        adata_sub,
        groupby='subcluster',
        method='t-test_overestim_var',
        key_added='L2_DE'
    )

    result = adata_sub.uns['L2_DE']
    subclusters = result['names'].dtype.names
    
    print("L2 raw exists:", adata_sub.raw is not None)

    first_subcl = subclusters[0]
    print("First L2 logFC values:",
      result['logfoldchanges'][first_subcl][:5])

    for subcl in subclusters:
        gene_tables[f"{subcl}"] = pd.DataFrame({
            'gene': result['names'][subcl],
            'logfoldchange': result['logfoldchanges'][subcl],
            'pval': result['pvals'][subcl],
            'pval_adj': result['pvals_adj'][subcl],
            'score': result['scores'][subcl],
        })

    del adata_sub
    gc.collect()


logger.info("Writing Excel workbook ...")
table_dir = "../results/13MANUSCRIPT_PLOTS_TABLES/tables/"
output_path = table_dir + "diff_exp_genes_L1_and_L2_clusters.xlsx"

with pd.ExcelWriter(output_path) as writer:
    for sheet_name, df in gene_tables.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

logger.info("All done.")
sys.exit(0)